# Chapter 3 · From Counting to a Network

A network that learns the bigram table of chapter 2 by correcting scores with gradient descent. Standard Python and CPU: there is nothing to install. Run the cells in order.

The full explanation is in the PDF chapter. This notebook is the lab: the same program, split into parts, so you can run it and experiment.

## 1. The data

The same names, the same held-out set and the same marks as in chapter 2. `transitions` turns the names into (context, output) pairs. The last line of constants gathers the decisions of the experiment: steps, learning rate and brake.

In [1]:
"""Chapter 3: a network that learns to count. Standard library and CPU only."""
import math
import random

TRAINING = ("ana", "anna", "alba", "alan")
HELD_OUT = ("lana", "bala")
START, END = "^", "$"
CONTEXTS = (START, "a", "b", "l", "n")
OUTPUTS = ("a", "b", "l", "n", END)
STEPS, RATE, BRAKE = 1000, 5.0, 0.05


def transitions(names):
    pairs = []
    for name in names:
        string = START + name + END
        pairs.extend(zip(string, string[1:]))
    return pairs

In [2]:
training = transitions(TRAINING)
held_out = transitions(HELD_OUT)
print(len(training), "training transitions and", len(held_out), "held-out ones")
print("Those of ana:", training[:4])

19 training transitions and 10 held-out ones
Those of ana: [('^', 'a'), ('a', 'n'), ('n', 'a'), ('a', '$')]


## 2. From scores to loss

`softmax` turns a row of scores into probabilities: the exponential of each one, divided by their sum. `loss` is the mean NLL of chapter 2: it receives probabilities, so it works both for the network and for the counted tables.

In [3]:
def softmax(scores):
    exponentials = [math.exp(z) for z in scores]
    total = sum(exponentials)
    return [e / total for e in exponentials]


def probabilities(table):
    return {c: softmax(table[c]) for c in CONTEXTS}


def loss(probs, pairs):
    costs = []
    for c, s in pairs:
        p = probs[c][OUTPUTS.index(s)]
        if p == 0:
            return math.inf
        costs.append(-math.log(p))
    return sum(costs) / len(costs)

Check the two examples from the chapter: the row from exercise 2 and a row of zeros, which gives the uniform model.

In [4]:
print("softmax([0, 0, 1, 2, 2]) =", [round(p, 3) for p in softmax([0, 0, 1, 2, 2])])
print("softmax([0, 0, 0, 0, 0]) =", [round(p, 3) for p in softmax([0, 0, 0, 0, 0])])

softmax([0, 0, 1, 2, 2]) = [0.051, 0.051, 0.139, 0.379, 0.379]
softmax([0, 0, 0, 0, 0]) = [0.2, 0.2, 0.2, 0.2, 0.2]


## 3. Learning

`corrections` applies the observed-output rule: it adds the probability to every output and subtracts 1 from the one that appeared. `step` updates each score with the brake: `(1 - rate * brake) * z - rate * correction`. `train` starts with all the scores at zero, that is, with the uniform model.

In [5]:
def corrections(table, pairs):
    grad = {c: [0.0] * len(OUTPUTS) for c in CONTEXTS}
    for c, s in pairs:
        p = softmax(table[c])
        for j, output in enumerate(OUTPUTS):
            observed = 1.0 if output == s else 0.0
            grad[c][j] += (p[j] - observed) / len(pairs)
    return grad


def step(table, pairs, rate, brake):
    grad = corrections(table, pairs)
    for c in CONTEXTS:
        table[c] = [(1 - rate * brake) * z - rate * g
                    for z, g in zip(table[c], grad[c])]


def train(pairs, steps=STEPS, rate=RATE, brake=0.0):
    table = {c: [0.0] * len(OUTPUTS) for c in CONTEXTS}
    history = [(0, loss(probabilities(table), pairs))]
    for n in range(1, steps + 1):
        step(table, pairs, rate, brake)
        if n in (1, 10, 100, 1000):
            history.append((n, loss(probabilities(table), pairs)))
    return table, history

Follow the first step for the row of `a` by hand and compare it with the chapter: the correction divided by 19 and the probabilities after one step with learning rate 5.

In [6]:
table = {c: [0.0] * len(OUTPUTS) for c in CONTEXTS}
print("Correction of the row of a:", [round(g, 3) for g in corrections(table, training)["a"]])
step(table, training, RATE, 0.0)
print("Scores after one step:", [round(z, 2) for z in table["a"]])
print("Probabilities after one step:", [round(p, 2) for p in softmax(table["a"])])

Correction of the row of a: [0.084, 0.084, -0.021, -0.074, -0.074]
Scores after one step: [-0.42, -0.42, 0.11, 0.37, 0.37]
Probabilities after one step: [0.12, 0.12, 0.21, 0.27, 0.27]


## 4. Comparing and generating

`by_counting` rebuilds the tables of chapter 2 as a reference. `generate` samples names from any probability table. `run` trains the two networks and shows all the results.

In [7]:
def by_counting(pairs, alpha):
    probs = {}
    for c in CONTEXTS:
        counts = [sum(1 for a, s in pairs if a == c and s == output)
                  for output in OUTPUTS]
        total = sum(counts) + alpha * len(OUTPUTS)
        probs[c] = [(n + alpha) / total for n in counts]
    return probs


def generate(probs, rng, max_steps=20):
    c, letters = START, []
    for _ in range(max_steps):
        s = rng.choices(OUTPUTS, weights=probs[c], k=1)[0]
        if s == END:
            return "".join(letters), True
        letters.append(s)
        c = s
    return "".join(letters), False

In [8]:
def run():
    training = transitions(TRAINING)
    held_out = transitions(HELD_OUT)
    net, history = train(training)
    net_brake, _ = train(training, brake=BRAKE)
    models = {
        "Uniform": {c: [1 / len(OUTPUTS)] * len(OUTPUTS) for c in CONTEXTS},
        "Unsmoothed counts": by_counting(training, 0),
        "Smoothed counts": by_counting(training, 1),
        "Network without brake": probabilities(net),
        "Network with brake": probabilities(net_brake),
    }
    print("Training loss:", history)
    for name, probs in models.items():
        print(f"{name:22} training {loss(probs, training):.6f}"
              f"  held-out {loss(probs, held_out):.6f}")
    for name in ("Unsmoothed counts", "Network without brake",
                 "Smoothed counts", "Network with brake"):
        row = [round(p, 3) for p in models[name]["a"]]
        print(f"Row of a, {name}: {row}")
    rng = random.Random(7)
    for _ in range(10):
        text, complete = generate(models["Network with brake"], rng)
        print(repr(text), "end" if complete else "truncated")
    return models, history


In [9]:
models, history = run()

Training loss: [(0, 1.6094379124341003), (1, 1.2917141766358304), (10, 0.85891603260276), (100, 0.7582590042945252), (1000, 0.7485585431242812)]
Uniform                training 1.609438  held-out 1.609438
Unsmoothed counts      training 0.747513  held-out inf
Smoothed counts        training 1.113961  held-out 1.409950
Network without brake  training 0.748559  held-out 2.355192
Network with brake     training 1.086813  held-out 1.438037
Row of a, Unsmoothed counts: [0.0, 0.0, 0.25, 0.375, 0.375]
Row of a, Network without brake: [0.0, 0.0, 0.25, 0.375, 0.375]
Row of a, Smoothed counts: [0.077, 0.077, 0.231, 0.308, 0.308]
Row of a, Network with brake: [0.084, 0.084, 0.22, 0.306, 0.306]
'ablanbananabbnall' end
'an' end
'a' end
'abalnannblaallbanba' end
'banl' end
'la' end
'annanannl' end
'a' end
'anbn' end
'ana' end


## 5. Experiment with the brake

Change the value of `brake` and run the cell again (exercise 10). Look at the row of `a` and at the training loss. Write down first what you expect to happen.

In [10]:
brake = 0.01  # also try 0.05 and 0.1
net, _ = train(training, brake=brake)
probs = probabilities(net)
print("Row of a:", [round(p, 3) for p in probs["a"]])
print("Training loss:", round(loss(probs, training), 4))

Row of a: [0.032, 0.032, 0.235, 0.35, 0.35]
Training loss: 0.8703


## Exercises

1. With a row of scores [0, 0, 0, 0, 0], compute the five probabilities. Which model of chapter 2 does it correspond to?

2. Compute the softmax of [0, 0, 1, 2, 2] without looking at the chapter's table. Check that the probabilities add up to one.

3. Add 3 to all the scores of the previous exercise and compute softmax again. What do you notice? Try to explain why this happens.

4. With all the scores at zero, write down the five corrections for the transition `a`→`n`. Which one is negative, and what will happen to its score?

5. Repeat the previous exercise if the network already gave 0.9 to `n` and 0.025 to each of the other outputs. What changes?

6. Reconstruct the total correction of the row of `a` in the first step, before and after dividing by 19.

7. Check with the row of `n` that the correction becomes zero when the probabilities match the counted proportions.

8. Why does the network without a brake never give a probability of exactly zero to starting with `l`? What consequence does this have for `lana`?

9. With a learning rate of 5 and a brake of 0.05, what value will a score of 2 have after one step if its correction is zero? And after many steps, if the correction stayed at zero?

10. Run the program with a brake of 0.01 and of 0.1. Describe what happens to the row of `a` and to the training loss. Do not use the held-out set to decide which one is better.

11. Compare the network with a brake and the smoothed table in the chapter's results table. How are they alike? Are they the same model?

12. In chapter 4 the network will look at three letters. Which pieces of this chapter will be kept, and which one will change?

## References

Teaching idea: the first lesson of [makemore](https://www.youtube.com/watch?v=PaCmpygFfXo), part of [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), by Andrej Karpathy. Here it has been rewritten in standard Python. The minimal name collection, the code and the explanations are original work.